# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [1]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [2]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF-TOKEN']
login(hf_token, add_to_git_credential=True)

In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [4]:
openai = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [5]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [6]:
len(fine_tune_train)

100

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [7]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [8]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [9]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [10]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [11]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [12]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [13]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [14]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [15]:
train_file

FileObject(id='file-Kq45c22GBaUc3Mw6XegyJK', bytes=55219, created_at=1787051774, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [16]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [17]:
validation_file

FileObject(id='file-WctBULQ4Uy4q7D35nfhDda', bytes=27686, created_at=1787052025, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [ ]:
# this fince-tune is deprectaed by openaI
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)
openai.fine_tuning.jobs.list(limit=1)
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
openai.fine_tuning.jobs.retrieve(job_id)
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

https://platform.openai.com/finetune


In [20]:
# Create a function to build our prompt history with examples
def build_few_shot_messages(examples):
    messages = [
        {"role": "system", "content": "You are a price estimation assistant. Respond with ONLY the price (e.g., $XX.XX) and absolutely no explanation."}
    ]
    # Add our training examples to the conversation history
    for item in examples:
        messages.extend(messages_for(item)) # Re-using your messages_for function from Cell 7
    return messages

In [23]:
# Select 10 examples from our training data to guide the model
few_shot_examples = fine_tune_train[:10]
base_messages = build_few_shot_messages(few_shot_examples)

# Let's look at the first few messages in our context
base_messages[:3]

[{'role': 'system',
  'content': 'You are a price estimation assistant. Respond with ONLY the price (e.g., $XX.XX) and absolutely no explanation.'},
 {'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [ ]:
# The updated inference function
def gpt_few_shot_pricer(item):
    # Start with our few-shot examples
    messages = list(base_messages)

    # Append the new product we want to test
    messages.append({
        "role": "user",
        "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    })

    # Call the base model directly
    response = openai.chat.completions.create(
        model="gpt-4.1-nano-2025-04-14", # Using the base model instead of a fine-tuned version
        messages=messages,
        max_tokens=10,
        temperature=0.0 # Keep temperature at 0 for deterministic pricing
    )
    return response.choices[0].message.content.strip()

In [25]:
# Try it out on the first test item
print("Actual Price:", test[0].price)
print("Predicted Price:", gpt_few_shot_pricer(test[0]))

Actual Price: 219.0
Predicted Price: $229.00


In [26]:
# Run the full evaluation!
evaluate(gpt_few_shot_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $74 $25 $20 $20 $80 $54 $25 $1 $69 $314 $20 $10 $24 $29 $3 $51 $4 $90 $29 $44 $36 $5 $55 $182 $253 $304 $10 $191 $60 $25 $30 $30 $50 $15 $170 $30 $26 $24 $18 $170 $50 $22 $65 $90 $5 $12 $2 $55 $12 $22 $100 $175 $10 $137 $44 $6 $50 $43 $1 $106 $28 $41 $20 $229 $10 $50 $255 $65 $54 $12 $13 $90 $3 $10 $20 $176 $3 $3 $1 $10 $8 $15 $74 $7 $10 $48 $116 $30 $1 $8 $5 $0 $15 $0 $98 $4 $33 $80 $285 $10 $27 $7 $49 $19 $282 $12 $355 $1 $70 $0 $86 $59 $38 $16 $60 $5 $0 $34 $13 $14 $510 $60 $36 $10 $30 $10 $61 $11 $89 $109 $53 $5 $0 $85 $5 $115 $30 $38 $52 $26 $101 $10 $11 $104 $58 $5 $290 $135 $3 $1 $193 $27 $50 $3 $129 $41 $36 $0 $0 $110 $17 $8 $0 $41 $7 $52 $20 $10 $10 $5 $8 $120 $3 $12 $81 $2 $37 $26 $3 $145 $25 $150 $59 $40 $8 $73 $3 $30 $7 $15 $39 $15 $11 $50 $70 $9 $20 $21 $4 

# Step 3

Test our fine tuned model

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [ ]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [ ]:
# Try this out

test_messages_for(test[0])

In [ ]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [ ]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)

In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000